# HyDE vs. SDCP-v2 Comparison

**Goal**: Compare SDCP-v2 against HyDE (Gao et al., ACL 2023) — the closest related inference-time query expansion method.

**HyDE**: LLM generates a hypothetical answer → embed that answer → retrieve using its embedding.

**SDCP-v2**: Generates P_pos + P_neg → contrastive reranking → retrieval.

Both are label-free, training-free, and add one extra LLM call. Key difference: SDCP uses
contrastive scoring (positive - negative), HyDE uses only the positive direction.

**Model**: Mistral-7B-Instruct-v0.2 (same as main experiments)  
**Datasets**: TruthfulQA (615Q) + MMLU (1596Q)  
**Expected time**: ~50 min TQA + ~130 min MMLU

In [1]:
# ── Cell 0: Load model + datasets ─────────────────────────────────────────────
import os, gc, time, json
import numpy as np
import torch, faiss
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer as rs_module
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime
from tqdm import tqdm
import pandas as pd

MODELS_DIR   = '/home/user/RAG_Best_Practices/models'
DATASETS_DIR = '/home/user/RAG_Best_Practices/datasets'
OUTPUT_DIR   = '/home/user/RAG_Best_Practices/outputs'

MISTRAL_LOCAL = f'{MODELS_DIR}/mistral-7b'
EMBED_PATH    = f'{MODELS_DIR}/minilm'
N_TRUTHFULQA     = 615
MMLU_PER_SUBJECT = 28
RANDOM_SEED      = 42
CHOICE_LABELS    = ['A', 'B', 'C', 'D']
SYS_MSG = 'You are a truthful expert question-answering bot and should correctly and concisely answer the following question'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gc.collect(); torch.cuda.empty_cache()

print('Loading Mistral-7B-Instruct (4-bit NF4)...')
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
llm = AutoModelForCausalLM.from_pretrained(
    MISTRAL_LOCAL, quantization_config=bnb, device_map='auto'
)
tokenizer = AutoTokenizer.from_pretrained(MISTRAL_LOCAL, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print('Loading MiniLM...')
embed_model = SentenceTransformer(EMBED_PATH).to(DEVICE)
print('Models ready!')

Loading Mistral-7B-Instruct (4-bit NF4)...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loading MiniLM...
Models ready!


In [2]:
# ── Cell 1: Load datasets + build KBs ─────────────────────────────────────────
print('Loading TruthfulQA...')
tqa_raw = load_from_disk(f'{DATASETS_DIR}/truthfulqa').to_pandas()
tqa_all = tqa_raw[['question','best_answer','correct_answers','incorrect_answers']].copy()
tqa_all['correct_answers']   = tqa_all['correct_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['incorrect_answers'] = tqa_all['incorrect_answers'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else [x])
tqa_all['best_answer']       = tqa_all['best_answer'].apply(lambda x: [x] if x else [])
tqa_all = tqa_all[(tqa_all['correct_answers'].apply(len) > 1) &
                   (tqa_all['incorrect_answers'].apply(len) > 1)].reset_index(drop=True)
tqa_test_idx = tqa_all.sample(n=N_TRUTHFULQA, random_state=RANDOM_SEED).index
tqa_kb_idx   = tqa_all.index.difference(tqa_test_idx)
tqa_test = tqa_all.loc[tqa_test_idx].reset_index(drop=True)
tqa_kb   = tqa_all.loc[tqa_kb_idx].reset_index(drop=True)
del tqa_raw; gc.collect()
print(f'  TQA test={len(tqa_test)}Q  KB={len(tqa_kb)}Q')

print('Loading MMLU...')
mmlu_raw = load_from_disk(f'{DATASETS_DIR}/mmlu').to_pandas()
def mmlu_to_unified(row):
    choices   = list(row['choices'])
    ans_idx   = int(row['answer'])
    correct   = choices[ans_idx]
    incorrect = [choices[i] for i in range(len(choices)) if i != ans_idx]
    formatted_q = (row['question'] + '\n' +
                   '\n'.join(f'{CHOICE_LABELS[i]}) {choices[i]}' for i in range(len(choices))))
    return pd.Series({'question': formatted_q, 'question_plain': row['question'],
                      'subject': row['subject'], 'best_answer': [correct],
                      'correct_answers': [correct], 'incorrect_answers': incorrect,
                      'answer_idx': ans_idx, 'choices': choices})
mmlu_test_parts, mmlu_kb_parts = [], []
for subject, group in mmlu_raw.groupby('subject'):
    group = group.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    n_test = min(MMLU_PER_SUBJECT, len(group))
    mmlu_test_parts.append(group.iloc[:n_test])
    mmlu_kb_parts.append(group.iloc[n_test:])
mmlu_test = pd.concat(mmlu_test_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
mmlu_kb   = pd.concat(mmlu_kb_parts).reset_index(drop=True).apply(mmlu_to_unified, axis=1)
del mmlu_raw; gc.collect()
print(f'  MMLU test={len(mmlu_test)}Q  KB={len(mmlu_kb)}Q')

# Build FAISS indexes
tqa_kb_embs = np.array(embed_model.encode(tqa_kb['question'].tolist(), batch_size=256), dtype=np.float32)
faiss.normalize_L2(tqa_kb_embs)
tqa_index = faiss.IndexFlatIP(tqa_kb_embs.shape[1]); tqa_index.add(tqa_kb_embs)

mmlu_kb_embs = np.array(embed_model.encode(mmlu_kb['question'].tolist(), batch_size=256), dtype=np.float32)
faiss.normalize_L2(mmlu_kb_embs)
mmlu_index = faiss.IndexFlatIP(mmlu_kb_embs.shape[1]); mmlu_index.add(mmlu_kb_embs)
print('KBs ready!')

Loading TruthfulQA...
  TQA test=615Q  KB=100Q
Loading MMLU...
  MMLU test=1596Q  KB=228Q
KBs ready!


In [3]:
# ── Cell 2: Shared utility functions ──────────────────────────────────────────
def make_prompt(user_content):
    return f'[INST] {SYS_MSG}\n{user_content} [/INST]'

def generate(prompts, max_new_tokens=25, num_beams=2):
    enc = tokenizer(prompts, return_tensors='pt', padding=True,
                    truncation=True, max_length=2048).to(DEVICE)
    input_len = enc['input_ids'].shape[1]
    with torch.no_grad():
        out = llm.generate(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            max_new_tokens=max_new_tokens, num_beams=num_beams,
            pad_token_id=tokenizer.eos_token_id,
        )
    return [tokenizer.decode(r[input_len:], skip_special_tokens=True).strip()
            or 'I have no comment' for r in out]

def clean_response(resp):
    for stop in ['\nQuestion:','\nQ:','\n---','\nIncorrect','\nCorrect',
                 '\nVERIFIED','\nExample','\n\n','\nMy initial','\nContext:',
                 '\nFor the question', '\nCommon']:
        if stop in resp: resp = resp[:resp.index(stop)]
    return resp.strip().strip('"').strip("'") or 'I have no comment'

def retrieve_hyde(hypo_doc, kb_df, kb_index, k_context=4):
    """HyDE: retrieve using embedding of hypothetical document."""
    h_emb = np.array(embed_model.encode([hypo_doc], show_progress_bar=False), dtype=np.float32)
    faiss.normalize_L2(h_emb)
    _, idxs = kb_index.search(h_emb, k_context)
    return ' '.join([kb_df.iloc[i]['question'] for i in idxs[0] if i < len(kb_df)])

def retrieve_base(query, kb_df, kb_index, k_context=4):
    """Standard RAG: retrieve using original query embedding."""
    q_emb = np.array(embed_model.encode([query], show_progress_bar=False), dtype=np.float32)
    faiss.normalize_L2(q_emb)
    _, idxs = kb_index.search(q_emb, k_context)
    return ' '.join([kb_df.iloc[i]['question'] for i in idxs[0] if i < len(kb_df)])

def compute_rouge1_per_query(generated, references):
    scorer = rs_module.RougeScorer(['rouge1'], use_stemmer=True)
    scores = []
    for gen, refs in zip(generated, references):
        best = 0
        for ref in refs:
            if not ref: continue
            s = scorer.score(ref, gen)
            best = max(best, s['rouge1'].fmeasure)
        scores.append(best * 100)
    return scores

def compute_accuracy(generated, dataset):
    correct = 0
    for gen, (_, row) in zip(generated, dataset.iterrows()):
        correct_text = row['best_answer'][0] if isinstance(row['best_answer'], list) else row['best_answer']
        ans_idx = int(row.get('answer_idx', -1))
        if ans_idx >= 0:
            label = CHOICE_LABELS[ans_idx]
            if label in gen[:5] or correct_text.lower() in gen.lower(): correct += 1
        else:
            if correct_text.lower() in gen.lower(): correct += 1
    return correct / len(generated) * 100

def bootstrap_test(scores_a, scores_b, n_boot=10000, seed=42):
    """Paired bootstrap significance test (Efron & Tibshirani)."""
    rng = np.random.default_rng(seed)
    diffs = np.array(scores_a) - np.array(scores_b)
    observed = np.mean(diffs)
    centered = diffs - observed
    boots = [np.mean(rng.choice(centered, size=len(centered), replace=True)) for _ in range(n_boot)]
    p_value = np.mean(np.array(boots) >= observed) if observed > 0 else np.mean(np.array(boots) <= observed)
    return observed, p_value

print('Utility functions ready.')

Utility functions ready.


In [4]:
# ── Cell 3: HyDE on TruthfulQA ────────────────────────────────────────────────
# HyDE: generate hypothetical answer → embed → retrieve → generate final answer
print('=== HyDE | Mistral-7B | TruthfulQA | 615Q ===')
t0 = time.time()
gen_hyde_tqa, refs_tqa, hypo_docs_tqa = [], [], []

for idx, row in tqdm(tqa_test.iterrows(), total=len(tqa_test), desc='HyDE/TQA'):
    query = row['question']
    best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

    # Step 1: generate hypothetical document (HyDE)
    hypo_prompt = f'Write a short factual passage that directly answers this question:\nQuestion: {query}\nPassage:'
    hypo_doc = clean_response(generate([make_prompt(hypo_prompt)], max_new_tokens=40)[0])

    # Step 2: retrieve using hypothetical doc embedding
    context = retrieve_hyde(hypo_doc, tqa_kb, tqa_index, k_context=4)

    # Step 3: final generation with retrieved context
    final_prompt = f'Context: {context}\nQuestion: {query}\nAnswer concisely:'
    final = clean_response(generate([make_prompt(final_prompt)], max_new_tokens=25)[0])

    gen_hyde_tqa.append(final)
    refs_tqa.append(best_answer)
    hypo_docs_tqa.append(hypo_doc)
    if idx % 30 == 0: gc.collect(); torch.cuda.empty_cache()

r1_per_q_hyde_tqa = compute_rouge1_per_query(gen_hyde_tqa, refs_tqa)
r1_hyde_tqa = np.mean(r1_per_q_hyde_tqa)
elapsed = (time.time() - t0) / 60
print(f'HyDE TQA: R1={r1_hyde_tqa:.2f}  (SDCP-v2 ref: 35.59)  [{elapsed:.1f} min]')

=== HyDE | Mistral-7B | TruthfulQA | 615Q ===


HyDE/TQA: 100%|██████████| 615/615 [3:30:26<00:00, 20.53s/it]  


HyDE TQA: R1=31.05  (SDCP-v2 ref: 35.59)  [210.4 min]


In [5]:
# ── Cell 4: Base RAG on TruthfulQA (for 3-way comparison) ─────────────────────
print('=== Base RAG | Mistral-7B | TruthfulQA | 615Q ===')
t0 = time.time()
gen_base_tqa = []

for idx, row in tqdm(tqa_test.iterrows(), total=len(tqa_test), desc='Base/TQA'):
    query = row['question']
    context = retrieve_base(query, tqa_kb, tqa_index, k_context=4)
    final_prompt = f'Context: {context}\nQuestion: {query}\nAnswer concisely:'
    final = clean_response(generate([make_prompt(final_prompt)], max_new_tokens=25)[0])
    gen_base_tqa.append(final)
    if idx % 30 == 0: gc.collect(); torch.cuda.empty_cache()

r1_per_q_base_tqa = compute_rouge1_per_query(gen_base_tqa, refs_tqa)
r1_base_tqa = np.mean(r1_per_q_base_tqa)
elapsed = (time.time() - t0) / 60
print(f'Base TQA: R1={r1_base_tqa:.2f}  [{elapsed:.1f} min]')

=== Base RAG | Mistral-7B | TruthfulQA | 615Q ===


Base/TQA: 100%|██████████| 615/615 [1:22:24<00:00,  8.04s/it]

Base TQA: R1=31.25  [82.4 min]


In [6]:
# ── Cell 5: HyDE on MMLU ──────────────────────────────────────────────────────
print('=== HyDE | Mistral-7B | MMLU | 1596Q ===')
t0 = time.time()
gen_hyde_mmlu, refs_mmlu, hypo_docs_mmlu = [], [], []

for idx, row in tqdm(mmlu_test.iterrows(), total=len(mmlu_test), desc='HyDE/MMLU'):
    query   = row['question']
    q_plain = row['question_plain']
    best_answer = row['best_answer'] if isinstance(row['best_answer'], list) else [row['best_answer']]

    hypo_prompt = f'Write a short factual passage that directly answers this question:\nQuestion: {q_plain}\nPassage:'
    hypo_doc = clean_response(generate([make_prompt(hypo_prompt)], max_new_tokens=40)[0])

    context = retrieve_hyde(hypo_doc, mmlu_kb, mmlu_index, k_context=4)

    final_prompt = f'Context: {context}\nQuestion: {query}\nAnswer concisely:'
    final = clean_response(generate([make_prompt(final_prompt)], max_new_tokens=25)[0])

    gen_hyde_mmlu.append(final)
    refs_mmlu.append(best_answer)
    hypo_docs_mmlu.append(hypo_doc)
    if idx % 30 == 0: gc.collect(); torch.cuda.empty_cache()

r1_per_q_hyde_mmlu = compute_rouge1_per_query(gen_hyde_mmlu, refs_mmlu)
r1_hyde_mmlu = np.mean(r1_per_q_hyde_mmlu)
acc_hyde_mmlu = compute_accuracy(gen_hyde_mmlu, mmlu_test)
elapsed = (time.time() - t0) / 60
print(f'HyDE MMLU: R1={r1_hyde_mmlu:.2f}  Acc={acc_hyde_mmlu:.1f}%  (SDCP-v2 ref: R1=35.44, Acc=45.0%)  [{elapsed:.1f} min]')

=== HyDE | Mistral-7B | MMLU | 1596Q ===


HyDE/MMLU: 100%|██████████| 1596/1596 [9:48:49<00:00, 22.14s/it]  


HyDE MMLU: R1=30.74  Acc=43.0%  (SDCP-v2 ref: R1=35.44, Acc=45.0%)  [588.8 min]


In [7]:
# ── Cell 6: Statistical significance + summary ────────────────────────────────
# Load SDCP-v2 per-query scores from saved CSV
import glob
sdcp_tqa_files = sorted(glob.glob(f'{OUTPUT_DIR}/sdcp_v2_wiki_tqa_*.csv'))
# Use the main SDCP-v2 results (not wiki KB) — check for matching file
# If Mistral per-query CSV exists, use it; otherwise note manual entry
print('Available SDCP output files:')
for f in glob.glob(f'{OUTPUT_DIR}/sdcp_v2*.csv'): print(' ', f)

# Bootstrap test: HyDE vs Base on TQA
diff_hb_tqa, p_hb_tqa = bootstrap_test(r1_per_q_hyde_tqa, r1_per_q_base_tqa)

# Summary table
sdcp_r1_tqa  = 35.59  # from paper
sdcp_r1_mmlu = 35.44
sdcp_acc_mmlu = 45.0

print('\n' + '=' * 70)
print('COMPARISON: Base RAG  vs  HyDE  vs  SDCP-v2')
print('=' * 70)
print(f'{"Method":<25} {"TQA R1":>8} {"MMLU R1":>8} {"MMLU Acc":>10}')
print('-' * 55)
print(f'{"Base RAG":<25} {r1_base_tqa:>8.2f} {"--":>8} {"--":>10}')
print(f'{"HyDE":<25} {r1_hyde_tqa:>8.2f} {r1_hyde_mmlu:>8.2f} {acc_hyde_mmlu:>9.1f}%')
print(f'{"SDCP-v2 (ref)":<25} {sdcp_r1_tqa:>8.2f} {sdcp_r1_mmlu:>8.2f} {sdcp_acc_mmlu:>9.1f}%')
print('=' * 70)
print(f'Bootstrap test (HyDE vs Base, TQA): delta={diff_hb_tqa:+.3f} R1, p={p_hb_tqa:.4f}')

# Save
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
results = {
    'Base_TQA_R1':       round(r1_base_tqa, 3),
    'HyDE_TQA_R1':       round(r1_hyde_tqa, 3),
    'HyDE_MMLU_R1':      round(r1_hyde_mmlu, 3),
    'HyDE_MMLU_Acc':     round(acc_hyde_mmlu, 3),
    'SDCP_v2_TQA_R1':    sdcp_r1_tqa,
    'SDCP_v2_MMLU_R1':   sdcp_r1_mmlu,
    'SDCP_v2_MMLU_Acc':  sdcp_acc_mmlu,
    'bootstrap_HyDE_vs_Base_TQA': {'delta_R1': round(diff_hb_tqa, 4), 'p_value': round(p_hb_tqa, 4)},
}
out_path = f'{OUTPUT_DIR}/hyde_comparison_{ts}.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2)

pd.DataFrame({
    'question': tqa_test['question'].tolist(),
    'base_generated': gen_base_tqa,
    'hyde_generated': gen_hyde_tqa,
    'hyde_hypo_doc': hypo_docs_tqa,
    'reference': [r[0] for r in refs_tqa],
    'r1_base': r1_per_q_base_tqa,
    'r1_hyde': r1_per_q_hyde_tqa,
}).to_csv(f'{OUTPUT_DIR}/hyde_tqa_perquery_{ts}.csv', index=False)

print(f'\nSaved to: {out_path}')
print('Next: add HyDE row to paper Table 1 and Related Work section.')

Available SDCP output files:
  /home/user/RAG_Best_Practices/outputs/sdcp_v2_wiki_tqa_20260506_175450.csv
  /home/user/RAG_Best_Practices/outputs/sdcp_v2_wiki_mmlu_20260506_175450.csv

COMPARISON: Base RAG  vs  HyDE  vs  SDCP-v2
Method                      TQA R1  MMLU R1   MMLU Acc
-------------------------------------------------------
Base RAG                     31.25       --         --
HyDE                         31.05    30.74      43.0%
SDCP-v2 (ref)                35.59    35.44      45.0%
Bootstrap test (HyDE vs Base, TQA): delta=-0.204 R1, p=0.3396

Saved to: /home/user/RAG_Best_Practices/outputs/hyde_comparison_20260508_193452.json
Next: add HyDE row to paper Table 1 and Related Work section.
